# Coconut Leaf Health Detection Model v2
### Color-Aware Classification: Healthy (Green) vs Unhealthy (Yellowing)

**Dataset:**
- `healthy-leaves/` — Green, healthy coconut leaves
- `unhealthy-yellowing/` — Yellowing, unhealthy coconut leaves

**Key Design Decisions:**
- **NO color augmentation** (brightness/hue changes corrupt the green vs yellow label)
- **Geometric augmentation only** (rotation, flip, zoom, shift)
- **EfficientNetB0** — ImageNet pretrained weights already encode color features well
- **Label Smoothing 0.1** — prevents 100% overconfidence (v1 issue fixed)
- **Data re-split 70/15/15** — ensures 500+ test images (supervisor requirement)

**Supervisor Requirements:**
- [ ] Test set ≥ 500 images
- [ ] Precision, Recall, F1 close to each other per class
- [ ] Similar values across all classes
- [ ] Target accuracy ≥ 95%

## 1. Setup and Imports

In [ ]:
import os
import shutil
import json
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

print(f"TensorFlow: {tf.__version__}")
print(f"GPU:        {tf.config.list_physical_devices('GPU')}")

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

## 2. Configuration

In [ ]:
BASE_DIR = os.path.abspath('..')

# Source data (flat image folders — no subfolders)
HEALTHY_SRC   = os.path.join(BASE_DIR, 'data', 'raw', 'healthy-leaves')
UNHEALTHY_SRC = os.path.join(BASE_DIR, 'data', 'raw', 'unhealthy-yellowing')

# New dataset for v2
DATASET_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'leaf_health_v2', 'dataset')
MODEL_DIR   = os.path.join(BASE_DIR, 'models', 'leaf_health_v2')
os.makedirs(MODEL_DIR, exist_ok=True)

IMG_SIZE   = 224
BATCH_SIZE = 32

PHASE1_EPOCHS = 25
PHASE2_EPOCHS = 20
LR_PHASE1     = 1e-3
LR_PHASE2     = 5e-5

# 70 / 15 / 15 split
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

CLASS_NAMES = ['healthy', 'unhealthy']  # alphabetical = ImageDataGenerator order

print(f"Healthy source:   {HEALTHY_SRC}")
print(f"Unhealthy source: {UNHEALTHY_SRC}")
print(f"Dataset dir:      {DATASET_DIR}")
print(f"Model dir:        {MODEL_DIR}")
print(f"Split:            {int(TRAIN_RATIO*100)} / {int(VAL_RATIO*100)} / {int(TEST_RATIO*100)}")

## 3. Pool All Data and Re-split

Collect all images from `training/`, `test/`, `validation/` sub-folders and re-split cleanly.

```
healthy-leaves:    4804 images  →  train ~3363 | val ~721 | test ~720
unhealthy-yellowing: 4181 images  →  train ~2927 | val ~627 | test ~627
                                                         ─────────────
                                   Total test:          ~1347  ✓ (≥500)
```

In [ ]:
VALID_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def collect_all_images(src_dir):
    """Walk all sub-folders and collect image paths."""
    images = []
    for root, _, files in os.walk(src_dir):
        for f in files:
            if f.lower().endswith(VALID_EXT):
                images.append(os.path.join(root, f))
    return images

healthy_all   = collect_all_images(HEALTHY_SRC)
unhealthy_all = collect_all_images(UNHEALTHY_SRC)

random.shuffle(healthy_all)
random.shuffle(unhealthy_all)

print(f"Healthy images:   {len(healthy_all)}")
print(f"Unhealthy images: {len(unhealthy_all)}")
print(f"Total:            {len(healthy_all) + len(unhealthy_all)}")

In [ ]:
def split_list(lst, train_r, val_r):
    n = len(lst)
    t = int(n * train_r)
    v = int(n * (train_r + val_r))
    return lst[:t], lst[t:v], lst[v:]

h_train, h_val, h_test = split_list(healthy_all,   TRAIN_RATIO, VAL_RATIO)
u_train, u_val, u_test = split_list(unhealthy_all, TRAIN_RATIO, VAL_RATIO)

test_total = len(h_test) + len(u_test)
print(f"{'Split':<8} {'Healthy':>10} {'Unhealthy':>12} {'Total':>8}")
print("-" * 44)
print(f"{'Train':<8} {len(h_train):>10} {len(u_train):>12} {len(h_train)+len(u_train):>8}")
print(f"{'Val':<8} {len(h_val):>10} {len(u_val):>12} {len(h_val)+len(u_val):>8}")
print(f"{'Test':<8} {len(h_test):>10} {len(u_test):>12} {test_total:>8}")
print("-" * 44)
print(f"\nTest ≥ 500: {test_total}  {'✓' if test_total >= 500 else '✗ NEED MORE DATA'}")

In [ ]:
def build_dataset_folders(splits_dict, dataset_dir):
    for split, classes in splits_dict.items():
        for cls, file_list in classes.items():
            dest = os.path.join(dataset_dir, split, cls)
            os.makedirs(dest, exist_ok=True)

            existing = len([f for f in os.listdir(dest) if f.lower().endswith(VALID_EXT)])
            if existing == len(file_list):
                print(f"  {split}/{cls}: {existing} files already present (skip)")
                continue

            for f in os.listdir(dest):
                os.remove(os.path.join(dest, f))

            for i, src in enumerate(file_list):
                ext = os.path.splitext(src)[1].lower() or '.jpg'
                shutil.copy2(src, os.path.join(dest, f"{cls}_{split}_{i:05d}{ext}"))

            print(f"  {split}/{cls}: copied {len(file_list)} files")

print("Building dataset structure...")
build_dataset_folders(
    {
        'train': {'healthy': h_train, 'unhealthy': u_train},
        'val':   {'healthy': h_val,   'unhealthy': u_val},
        'test':  {'healthy': h_test,  'unhealthy': u_test},
    },
    DATASET_DIR
)
print("Done!")

## 4. Dataset Summary

In [ ]:
data_summary = {}
print("=" * 55)
print("DATASET SUMMARY")
print("=" * 55)
for split in ['train', 'val', 'test']:
    data_summary[split] = {}
    total = 0
    print(f"\n{split.upper()}:")
    for cls in CLASS_NAMES:
        path = os.path.join(DATASET_DIR, split, cls)
        count = len([f for f in os.listdir(path) if f.lower().endswith(VALID_EXT)])
        data_summary[split][cls] = count
        total += count
        print(f"  {cls:<15} {count:>6}")
    print(f"  {'TOTAL':<15} {total:>6}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Class Distribution — Leaf Health v2', fontsize=13, fontweight='bold')
colors = ['#27ae60', '#e67e22']
for idx, split in enumerate(['train', 'val', 'test']):
    counts = [data_summary[split][c] for c in CLASS_NAMES]
    axes[idx].bar(CLASS_NAMES, counts, color=colors, edgecolor='black', linewidth=0.5)
    axes[idx].set_title(split.upper(), fontweight='bold')
    axes[idx].set_ylabel('Images')
    for i, v in enumerate(counts):
        axes[idx].text(i, v + 15, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
test_total = sum(data_summary['test'].values())
print(f"\nTest total: {test_total}  {'✓ (≥500)' if test_total >= 500 else '✗'}")

## 5. Color Analysis: Green vs Yellow

Visualize the HSV color distribution to confirm that healthy (green) and unhealthy (yellow) leaves are separable by color.

In [ ]:
def rgb_to_hsv_np(rgb_float):
    """
    Vectorized RGB→HSV conversion using numpy only (no cv2 needed).
    Input:  (H, W, 3) float32 in [0, 1]
    Output: h [0-180], s [0-255], v [0-255]  (OpenCV-compatible scale)
    """
    r, g, b = rgb_float[..., 0], rgb_float[..., 1], rgb_float[..., 2]
    cmax  = np.maximum(np.maximum(r, g), b)
    cmin  = np.minimum(np.minimum(r, g), b)
    delta = cmax - cmin

    # Hue
    h = np.zeros_like(r)
    m = delta > 0
    mr = m & (cmax == r);  h[mr] = 60 * (((g[mr] - b[mr]) / delta[mr]) % 6)
    mg = m & (cmax == g);  h[mg] = 60 * ((b[mg] - r[mg]) / delta[mg] + 2)
    mb = m & (cmax == b);  h[mb] = 60 * ((r[mb] - g[mb]) / delta[mb] + 4)
    h = h / 2.0  # scale to [0, 180]

    # Saturation
    s = np.where(cmax > 0, delta / cmax, 0) * 255

    # Value
    v = cmax * 255

    return h, s, v


def get_color_stats(img_dir, n_samples=100):
    """Sample images and compute mean HSV channel values (PIL-based, no cv2)."""
    files = [f for f in os.listdir(img_dir) if f.lower().endswith(VALID_EXT)]
    samples = random.sample(files, min(n_samples, len(files)))
    hues, sats, vals = [], [], []
    for fname in samples:
        try:
            img = Image.open(os.path.join(img_dir, fname)).convert('RGB').resize((112, 112))
            rgb = np.array(img, dtype=np.float32) / 255.0
            h, s, v = rgb_to_hsv_np(rgb)
            mask = s > 30  # skip near-white/grey background pixels
            if mask.sum() == 0:
                continue
            hues.append(h[mask].mean())
            sats.append(s[mask].mean())
            vals.append(v[mask].mean())
        except Exception:
            continue
    return np.array(hues), np.array(sats), np.array(vals)


print("Analyzing color distributions (PIL-based, ~30 sec)...")
h_hue, h_sat, h_val = get_color_stats(os.path.join(DATASET_DIR, 'train', 'healthy'),   150)
u_hue, u_sat, u_val = get_color_stats(os.path.join(DATASET_DIR, 'train', 'unhealthy'), 150)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('HSV Color Distribution: Healthy (Green) vs Unhealthy (Yellow)',
             fontsize=13, fontweight='bold')

chart_data = [
    ('Hue  (0=Red · 30=Yellow · 60=Green · 90=Cyan)',  h_hue, u_hue,  0, 180),
    ('Saturation  (higher = more vivid color)',          h_sat, u_sat,  0, 255),
    ('Value / Brightness',                               h_val, u_val,  0, 255),
]
for ax, (title, hv, uv, xmin, xmax) in zip(axes, chart_data):
    ax.hist(hv, bins=30, color='#27ae60', alpha=0.75, label=f'Healthy   mean={hv.mean():.1f}', density=True)
    ax.hist(uv, bins=30, color='#e67e22', alpha=0.75, label=f'Unhealthy mean={uv.mean():.1f}', density=True)
    ax.set_title(title, fontsize=8)
    ax.set_xlim(xmin, xmax)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'color_analysis_hsv.png'), dpi=150, bbox_inches='tight')
plt.show()

hue_sep = abs(h_hue.mean() - u_hue.mean())
print(f"\nColor Separation:")
print(f"  Healthy   Hue: {h_hue.mean():.1f} ± {h_hue.std():.1f}  (green ≈ 35-85)")
print(f"  Unhealthy Hue: {u_hue.mean():.1f} ± {u_hue.std():.1f}  (yellow ≈ 10-35)")
print(f"  Separation:    {hue_sep:.1f} units  {'✓ Classes are color-separable' if hue_sep > 10 else '⚠ Low — check data'}")

## 6. Sample Images Visualization

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('Sample Training Images — Healthy (Green) vs Unhealthy (Yellowing)',
             fontsize=13, fontweight='bold')

cls_colors = {'healthy': '#27ae60', 'unhealthy': '#e67e22'}
for row, cls in enumerate(CLASS_NAMES):
    cls_dir = os.path.join(DATASET_DIR, 'train', cls)
    files = [f for f in os.listdir(cls_dir) if f.lower().endswith(VALID_EXT)]
    for col, fname in enumerate(random.sample(files, min(6, len(files)))):
        img = keras_image.load_img(os.path.join(cls_dir, fname), target_size=(IMG_SIZE, IMG_SIZE))
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls.upper(), color=cls_colors[cls],
                                      fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Data Generators

### ⚠️ CRITICAL: NO Color Augmentation
Since the model learns **green = healthy / yellow = unhealthy**, any color augmentation  
(brightness, hue, channel shift) would corrupt the labels and **destroy accuracy**.

Only **geometric augmentation** is applied:
- Rotation, flip, zoom, shift → simulate different drone angles
- No brightness, no hue, no saturation changes

In [ ]:
# Geometric-only augmentation (color preserved!)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.20,
    height_shift_range=0.20,
    horizontal_flip=True,
    vertical_flip=False,     # Leaves have up/down orientation
    zoom_range=0.20,
    shear_range=0.10,
    fill_mode='nearest'
    # NO brightness_range  ← would make healthy look unhealthy
    # NO channel_shift_range ← would corrupt color signal
)

eval_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=True,
    seed=42
)
val_gen = eval_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)
test_gen = eval_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

print(f"Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}")
print(f"Class indices: {train_gen.class_indices}")
print(f"Test ≥ 500: {'✓' if test_gen.samples >= 500 else '✗'}")

## 8. Class Weights

In [ ]:
train_labels = train_gen.classes
cw = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weight_dict = {i: w for i, w in enumerate(cw)}

print("Class Weights (balanced):")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  {i} ({cls:<12}): {class_weight_dict[i]:.4f}")

imbalance = max(len(h_train), len(u_train)) / min(len(h_train), len(u_train))
print(f"\nClass imbalance ratio: {imbalance:.2f}x")

## 9. Focal Loss + Label Smoothing

**Label Smoothing (0.1)** prevents overconfidence:
- v1 issue: model output 100% for healthy leaves
- With smoothing: max output capped at ~95% → realistic confidence scores

In [ ]:
def focal_loss_smoothed(gamma=2.0, alpha=0.25, label_smoothing=0.1):
    """
    Focal Loss with Label Smoothing.
    label_smoothing=0.1 → softens hard targets (1→0.95, 0→0.05)
    Prevents overconfident 100% predictions.
    """
    n_classes = 2
    def loss_fn(y_true, y_pred):
        y_smooth = y_true * (1.0 - label_smoothing) + (label_smoothing / n_classes)
        eps = tf.keras.backend.epsilon()
        y_pred = tf.keras.backend.clip(y_pred, eps, 1.0 - eps)
        ce = -y_smooth * tf.keras.backend.log(y_pred)
        fw = tf.keras.backend.pow(1.0 - y_pred, gamma)
        return tf.keras.backend.sum(alpha * fw * ce, axis=-1)
    return loss_fn

print("Focal Loss + Label Smoothing ready")
print("  gamma=2.0           → harder examples get more focus")
print("  alpha=0.25          → class balancing")
print("  label_smoothing=0.1 → no overconfident 100% outputs")

## 10. Build Model — EfficientNetB0

**Why EfficientNetB0 for color classification?**
- ImageNet pretrained → already learned green/yellow color features
- Compound scaling → better texture + color feature extraction
- Higher baseline than MobileNetV2

In [ ]:
def build_model():
    # EfficientNetB0 expects pixel values [0, 255]
    # Our generator outputs [0, 1] → rescale back
    base = EfficientNetB0(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                          include_top=False, weights='imagenet')
    base.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(scale=255.0)(inputs)   # [0,1] → [0,255]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.20)(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation='softmax')(x)

    return keras.Model(inputs, outputs), base

model, base_model = build_model()
print(f"Architecture:     EfficientNetB0 + custom head")
print(f"Base layers:      {len(base_model.layers)}")
print(f"Total params:     {model.count_params():,}")
model.summary()

## 11. Phase 1 — Frozen Base

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(LR_PHASE1),
    loss=focal_loss_smoothed(gamma=2.0, alpha=0.25, label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_p1 = [
    ModelCheckpoint(os.path.join(MODEL_DIR, 'phase1_best.keras'),
                    monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

print("=" * 65)
print("PHASE 1 — Frozen EfficientNetB0 (learning color features)")
print("=" * 65)

t0 = time.time()
history_p1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=PHASE1_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks_p1,
    verbose=1
)
p1_min = (time.time() - t0) / 60
best_p1 = max(history_p1.history['val_accuracy'])
print(f"\nPhase 1: {p1_min:.1f} min | Best val acc: {best_p1*100:.2f}%")

## 12. Phase 2 — Fine-tuning

In [ ]:
# Unfreeze last 50 layers
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 50
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(LR_PHASE2),
    loss=focal_loss_smoothed(gamma=2.0, alpha=0.25, label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_p2 = [
    ModelCheckpoint(os.path.join(MODEL_DIR, 'best_model.keras'),
                    monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=1),
]

print(f"Fine-tuning from layer {fine_tune_at} / {len(base_model.layers)}")
print("=" * 65)
print("PHASE 2 — Fine-tuning (color-sensitive layers)")
print("=" * 65)

t0 = time.time()
history_p2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=PHASE2_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks_p2,
    verbose=1
)
p2_min = (time.time() - t0) / 60
total_min = p1_min + p2_min
best_p2 = max(history_p2.history['val_accuracy'])
print(f"\nPhase 2: {p2_min:.1f} min | Total: {total_min:.1f} min | Best val acc: {best_p2*100:.2f}%")

## 13. Training History

In [ ]:
hist = {
    k: history_p1.history[k] + history_p2.history[k]
    for k in ['accuracy', 'val_accuracy', 'loss', 'val_loss']
}
p1_end = len(history_p1.history['accuracy']) - 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History — Leaf Health v2 (EfficientNetB0)', fontsize=13, fontweight='bold')
for ax, (metric, title) in zip(axes, [('accuracy', 'Accuracy'), ('loss', 'Loss')]):
    ax.plot(hist[metric],         label='Train', linewidth=2)
    ax.plot(hist[f'val_{metric}'], label='Val',   linewidth=2)
    ax.axvline(p1_end, color='red', linestyle='--', alpha=0.7, label='Fine-tune start')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

final_train_acc = hist['accuracy'][-1]
final_val_acc   = hist['val_accuracy'][-1]
gap = abs(final_train_acc - final_val_acc)
print(f"Train: {final_train_acc*100:.2f}%  Val: {final_val_acc*100:.2f}%  Gap: {gap*100:.2f}%")
print(f"Overfitting: {'✓ OK' if gap < 0.05 else '⚠ Minor' if gap < 0.10 else '⚠⚠ High'}")

## 14. Evaluate on Test Set

In [ ]:
best_model = keras.models.load_model(
    os.path.join(MODEL_DIR, 'best_model.keras'),
    custom_objects={'loss_fn': focal_loss_smoothed(gamma=2.0, alpha=0.25, label_smoothing=0.1)}
)

test_gen.reset()
preds   = best_model.predict(test_gen, verbose=1)
y_true  = test_gen.classes
y_pred  = np.argmax(preds, axis=1)
y_conf  = np.max(preds, axis=1)
test_acc = np.mean(y_true == y_pred)

print(f"\nTest samples: {len(y_true)}")
print(f"Test accuracy: {test_acc*100:.2f}%  {'✓ (≥95%)' if test_acc >= 0.95 else '⚠ below target'}")
print(f"Max confidence: {y_conf.max()*100:.2f}%  {'✓ (no 100%)' if y_conf.max() < 0.999 else '⚠'}")
print(f"Mean confidence: {y_conf.mean()*100:.2f}%")

## 15. Class-wise Metrics — Supervisor Requirements

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None)
macro_p = np.mean(precision)
macro_r = np.mean(recall)
macro_f = np.mean(f1)

print("=" * 90)
print("CLASS-WISE METRICS")
print("=" * 90)
print(f"\n{'Class':<15} {'Precision':>12} {'Recall':>12} {'F1-Score':>12} {'Support':>10}")
print("-" * 65)
for i, cls in enumerate(CLASS_NAMES):
    print(f"{cls:<15} {precision[i]*100:>11.2f}% {recall[i]*100:>11.2f}% {f1[i]*100:>11.2f}% {support[i]:>10}")
print("-" * 65)
print(f"{'Macro Avg':<15} {macro_p*100:>11.2f}% {macro_r*100:>11.2f}% {macro_f*100:>11.2f}%")
print("=" * 90)

print("\n" + "=" * 90)
print("SUPERVISOR REQUIREMENT CHECKS")
print("=" * 90)

# Check 1: P/R/F1 close within each class
print("\n[1] P, R, F1 close to each other per class (max diff < 10%):")
for i, cls in enumerate(CLASS_NAMES):
    p, r, f = precision[i], recall[i], f1[i]
    max_d = max(abs(p-r), abs(p-f), abs(r-f))
    ok = max_d < 0.10
    print(f"  {cls.upper():12} P={p*100:.2f}%  R={r*100:.2f}%  F1={f*100:.2f}%  "
          f"max_diff={max_d*100:.2f}%  {'✓' if ok else '⚠'}")

# Check 2: Similar across classes
f1_diff = abs(f1[0] - f1[1])
p_diff  = abs(precision[0] - precision[1])
r_diff  = abs(recall[0] - recall[1])
print(f"\n[2] Similar values across both classes (diff < 10%):")
print(f"  F1 diff:        {f1_diff*100:.2f}%  {'✓' if f1_diff < 0.10 else '⚠'}")
print(f"  Precision diff: {p_diff*100:.2f}%  {'✓' if p_diff < 0.10 else '⚠'}")
print(f"  Recall diff:    {r_diff*100:.2f}%  {'✓' if r_diff < 0.10 else '⚠'}")

# Check 3: Test size
print(f"\n[3] Test set size: {len(y_true)}  {'✓ (≥500)' if len(y_true) >= 500 else '✗'}")

# Check 4: Accuracy target
print(f"\n[4] Accuracy: {test_acc*100:.2f}%  {'✓ (≥95%)' if test_acc >= 0.95 else '⚠ below 95% target'}")
print("=" * 90)

## 16. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Confusion Matrix — Leaf Health v2', fontsize=13, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Counts'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

cm_pct = cm.astype(float) / cm.sum(axis=1)[:, None] * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='RdYlGn',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1],
            cbar_kws={'label': '%'})
axes[1].set_title('Percentages (%)'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

for i, tc in enumerate(CLASS_NAMES):
    for j, pc in enumerate(CLASS_NAMES):
        sym = '✓' if i==j else '✗'
        print(f"  {sym} True:{tc:<12} Pred:{pc:<12} {cm[i,j]:>5} ({cm_pct[i,j]:.1f}%)")

## 17. Full Classification Report

In [ ]:
print("=" * 70)
print("FULL CLASSIFICATION REPORT")
print("=" * 70)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

# Confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Confidence Distribution — v2 (Label Smoothing Applied)', fontsize=12, fontweight='bold')

correct_c = y_conf[y_true == y_pred]
wrong_c   = y_conf[y_true != y_pred]

axes[0].hist(correct_c, bins=25, color='#27ae60', edgecolor='black', alpha=0.8)
axes[0].axvline(correct_c.mean(), color='darkgreen', linestyle='--',
                label=f'Mean: {correct_c.mean():.3f}')
axes[0].axvline(1.0, color='red', linestyle=':', alpha=0.5, label='100% line')
axes[0].set_title(f'Correct (n={len(correct_c)})'); axes[0].legend()

if len(wrong_c):
    axes[1].hist(wrong_c, bins=25, color='#e74c3c', edgecolor='black', alpha=0.8)
    axes[1].axvline(wrong_c.mean(), color='darkred', linestyle='--',
                    label=f'Mean: {wrong_c.mean():.3f}')
    axes[1].set_title(f'Wrong (n={len(wrong_c)})'); axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'No wrong predictions!', ha='center', va='center',
                 fontsize=14, color='green', transform=axes[1].transAxes)
    axes[1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confidence_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Max confidence: {y_conf.max()*100:.2f}%  {'✓ No 100%' if y_conf.max() < 0.999 else '⚠'}")

## 18. Sample Predictions

In [ ]:
filenames    = test_gen.filenames
correct_idx  = [i for i in range(len(y_true)) if y_true[i] == y_pred[i]]
wrong_idx    = [i for i in range(len(y_true)) if y_true[i] != y_pred[i]]
print(f"Total: {len(y_true)} | Correct: {len(correct_idx)} | Wrong: {len(wrong_idx)}")

def show_grid(indices, title, color, n=10):
    n = min(n, len(indices))
    if n == 0:
        print(f"{title}: nothing to show")
        return
    rows = max(1, (n + 4) // 5)
    fig, axes = plt.subplots(rows, 5, figsize=(15, 3.5 * rows))
    fig.suptitle(title, fontsize=13, fontweight='bold', color=color)
    if rows == 1:
        axes = axes.reshape(1, -1)
    for idx, i in enumerate(random.sample(indices, n)):
        r, c = idx // 5, idx % 5
        img = keras_image.load_img(
            os.path.join(DATASET_DIR, 'test', filenames[i]), target_size=(IMG_SIZE, IMG_SIZE))
        axes[r, c].imshow(img)
        axes[r, c].axis('off')
        axes[r, c].set_title(
            f'True: {CLASS_NAMES[y_true[i]]}\nPred: {CLASS_NAMES[y_pred[i]]} ({y_conf[i]*100:.1f}%)',
            fontsize=7, color=color)
    for idx in range(n, rows * 5):
        axes[idx // 5, idx % 5].axis('off')
    plt.tight_layout()
    fname = 'correct_predictions.png' if 'Correct' in title else 'wrong_predictions.png'
    plt.savefig(os.path.join(MODEL_DIR, fname), dpi=150, bbox_inches='tight')
    plt.show()

show_grid(correct_idx, 'CORRECT Predictions', 'green')
show_grid(wrong_idx,   f'WRONG Predictions ({len(wrong_idx)} total)', 'red')

## 19. Save Model Info

In [ ]:
model_info = {
    'model_name':   'leaf_health_v2',
    'architecture': 'EfficientNetB0',
    'version':      'v2',
    'classes':      CLASS_NAMES,
    'num_classes':  2,
    'input_shape':  [IMG_SIZE, IMG_SIZE, 3],
    'key_design_decisions': [
        'NO color augmentation — preserves green vs yellow color signal',
        'Geometric augmentation only (rotation/flip/zoom/shift)',
        'EfficientNetB0 — better color feature extraction than MobileNetV2',
        'Label smoothing 0.1 — fixes overconfident 100% predictions',
        'Data re-split 70/15/15 — ensures 500+ test images'
    ],
    'training': {
        'phase1_epochs':         PHASE1_EPOCHS,
        'phase2_epochs':         PHASE2_EPOCHS,
        'batch_size':            BATCH_SIZE,
        'lr_phase1':             LR_PHASE1,
        'lr_phase2':             LR_PHASE2,
        'loss_function':         'Focal Loss (gamma=2.0, alpha=0.25) + Label Smoothing 0.1',
        'optimizer':             'Adam',
        'augmentation':          'Geometric only — NO color changes',
        'class_weights':         {str(k): float(v) for k, v in class_weight_dict.items()},
        'training_time_minutes': round(total_min, 1),
        'final_train_accuracy':  float(final_train_acc),
        'final_val_accuracy':    float(final_val_acc)
    },
    'data': {
        'healthy_source':   'healthy-leaves/',
        'unhealthy_source': 'unhealthy-yellowing/',
        'split':            '70% train / 15% val / 15% test',
        'train_samples':    train_gen.samples,
        'val_samples':      val_gen.samples,
        'test_samples':     test_gen.samples,
        'train_healthy':    data_summary['train']['healthy'],
        'train_unhealthy':  data_summary['train']['unhealthy']
    },
    'test_performance': {
        'accuracy':            float(test_acc),
        'macro_precision':     float(macro_p),
        'macro_recall':        float(macro_r),
        'macro_f1':            float(macro_f),
        'healthy_precision':   float(precision[0]),
        'healthy_recall':      float(recall[0]),
        'healthy_f1':          float(f1[0]),
        'unhealthy_precision': float(precision[1]),
        'unhealthy_recall':    float(recall[1]),
        'unhealthy_f1':        float(f1[1])
    },
    'supervisor_checks': {
        'test_samples_500_plus':   bool(test_gen.samples >= 500),
        'accuracy_95_plus':        bool(test_acc >= 0.95),
        'healthy_prf_balanced':    bool(max(abs(precision[0]-recall[0]),abs(precision[0]-f1[0]),abs(recall[0]-f1[0])) < 0.10),
        'unhealthy_prf_balanced':  bool(max(abs(precision[1]-recall[1]),abs(precision[1]-f1[1]),abs(recall[1]-f1[1])) < 0.10),
        'cross_class_balanced':    bool(abs(f1[0]-f1[1]) < 0.10),
        'no_overconfidence':       bool(float(y_conf.max()) < 0.999)
    }
}

with open(os.path.join(MODEL_DIR, 'model_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)

print("Saved model_info.json")
print("\nFiles:")
for fname in sorted(os.listdir(MODEL_DIR)):
    kb = os.path.getsize(os.path.join(MODEL_DIR, fname)) / 1024
    print(f"  {fname:<45} {kb:>8.1f} KB")

## 20. Final Summary

In [ ]:
chk = model_info['supervisor_checks']

print("\n" + "=" * 85)
print("  COCONUT LEAF HEALTH MODEL v2 — FINAL SUMMARY")
print("=" * 85)
print()
print("  Dataset:")
print(f"    healthy-leaves/    +  unhealthy-yellowing/")
print(f"    Split 70/15/15 → Train:{train_gen.samples} | Val:{val_gen.samples} | Test:{test_gen.samples}")
print()
print("  Model:")
print(f"    Architecture:   EfficientNetB0 (ImageNet)")
print(f"    Loss:           Focal Loss + Label Smoothing 0.1")
print(f"    Augmentation:   Geometric only (NO color changes)")
print(f"    Training time:  {total_min:.1f} minutes")
print()
print("  Test Performance:")
print(f"    Accuracy:        {test_acc*100:.2f}%  {'✓' if test_acc >= 0.95 else '⚠'}")
print(f"    Macro Precision: {macro_p*100:.2f}%")
print(f"    Macro Recall:    {macro_r*100:.2f}%")
print(f"    Macro F1-Score:  {macro_f*100:.2f}%")
print()
print("  Class-wise:")
for i, cls in enumerate(CLASS_NAMES):
    mx = max(abs(precision[i]-recall[i]), abs(precision[i]-f1[i]), abs(recall[i]-f1[i]))
    print(f"    {cls.upper():12}  P={precision[i]*100:.2f}%  R={recall[i]*100:.2f}%  "
          f"F1={f1[i]*100:.2f}%  {'✓' if mx < 0.10 else '⚠'}")
print()
print("  Supervisor Requirement Checks:")
print(f"    [{'✓' if chk['test_samples_500_plus'] else '✗'}] Test set ≥ 500:          {test_gen.samples}")
print(f"    [{'✓' if chk['accuracy_95_plus'] else '⚠'}] Accuracy ≥ 95%:         {test_acc*100:.2f}%")
print(f"    [{'✓' if chk['healthy_prf_balanced'] else '⚠'}] Healthy  P/R/F1 balanced")
print(f"    [{'✓' if chk['unhealthy_prf_balanced'] else '⚠'}] Unhealthy P/R/F1 balanced")
print(f"    [{'✓' if chk['cross_class_balanced'] else '⚠'}] Cross-class balanced")
print(f"    [{'✓' if chk['no_overconfidence'] else '⚠'}] No overconfidence (max={y_conf.max()*100:.1f}%)")
print()
print(f"  Model saved: {MODEL_DIR}/best_model.keras")
print()
print("=" * 85)
print("              TRAINING COMPLETE — leaf_health_v2")
print("=" * 85)